**Script Generator**

In [0]:
import json, random, time, os
from datetime import datetime, UTC

# Use a Databricks volume or DBFS path — adjust to your workspace setup
output_path = "/Volumes/workspace/default/demo_series1/streaming_events/"
os.makedirs(output_path, exist_ok=True)

event_types = ["page_view", "add_to_cart", "purchase"]
products = ["laptop", "phone", "headphones", "monitor", "keyboard"]

def generate_batch(n=20):
    return [
        {
            "user_id": f"user_{random.randint(1, 200)}",
            "event_type": random.choice(event_types),
            "product": random.choice(products),
            "price": round(random.uniform(10, 1500), 2),
            "event_time": datetime.now(UTC).isoformat()
        }
        for _ in range(n)
    ]

for i in range(30):  # writes 30 batches, one every 5 sec
    batch = generate_batch()
    filename = f"{output_path}events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for record in batch:
            f.write(json.dumps(record) + "\n")
    print(f"Wrote {filename}")
    time.sleep(5)

Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779801.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779806.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779811.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779816.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779821.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779826.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779832.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779837.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779842.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779847.json
Wrote /Volumes/workspace/default/demo_series1/streaming_events/events_1785779852.json
Wrote /Volumes/workspace/default/demo_series1/streamin

**Stream with Autoloader**

In [0]:
from pyspark.sql.types import StructType, StringType, DoubleType

schema = (
    StructType()
    .add("user_id", StringType())
    .add("event_type", StringType())
    .add("product", StringType())
    .add("price", DoubleType())
    .add("event_time", StringType())
)

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load("/Volumes/workspace/default/demo_series1/streaming_events/")
)

query = (
    stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/demo_series1/checkpoints/bronze_events/")
    .outputMode("append")
    .trigger(availableNow=True)
    .table("bronze_streaming_events")
)

**Watch IT Land**

In [0]:
spark.sql("SELECT COUNT(*) FROM bronze_streaming_events").display()

COUNT(*)
1800


In [0]:
spark.sql("SELECT * FROM bronze_streaming_events ORDER BY event_time DESC LIMIT 10").display()

user_id,event_type,product,price,event_time
user_128,add_to_cart,monitor,146.09,2026-08-03T17:59:09.517016+00:00
user_166,page_view,headphones,223.05,2026-08-03T17:59:09.517011+00:00
user_197,page_view,phone,598.31,2026-08-03T17:59:09.517006+00:00
user_27,page_view,headphones,1311.87,2026-08-03T17:59:09.517000+00:00
user_114,purchase,laptop,397.79,2026-08-03T17:59:09.516996+00:00
user_127,page_view,keyboard,132.12,2026-08-03T17:59:09.516990+00:00
user_198,add_to_cart,laptop,416.25,2026-08-03T17:59:09.516985+00:00
user_71,purchase,phone,279.32,2026-08-03T17:59:09.516970+00:00
user_108,purchase,headphones,321.37,2026-08-03T17:59:09.516966+00:00
user_31,add_to_cart,monitor,924.54,2026-08-03T17:59:09.516961+00:00


**Read from Bronze as a stream**

In [0]:
bronze_stream = spark.readStream.table("bronze_streaming_events")

**Clean and transform**

In [0]:
from pyspark.sql.functions import col, to_timestamp, upper, trim

silver_stream = (
    bronze_stream
    .withColumn("event_time", to_timestamp(col("event_time")))
    .withColumn("user_id", trim(col("user_id")))
    .withColumn("event_type", trim(upper(col("event_type"))))
    .filter(col("price").isNotNull() & (col("price") > 0))
    .filter(col("user_id").isNotNull())
    .dropDuplicates(["user_id", "event_type", "product", "event_time"])
)

**Step 3 — Silver Delta table**

In [0]:
silver_query = (
    silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/demo_series1/checkpoints/silver_events/")
    .outputMode("append")
    .trigger(availableNow=True)
    .table("silver_events")
)

**Verify**

In [0]:
spark.sql("SELECT COUNT(*) FROM silver_events").display()
spark.sql("SELECT * FROM silver_events ORDER BY event_time DESC LIMIT 10").display()

COUNT(*)
1800


user_id,event_type,product,price,event_time
user_128,ADD_TO_CART,monitor,146.09,2026-08-03T17:59:09.517Z
user_166,PAGE_VIEW,headphones,223.05,2026-08-03T17:59:09.517Z
user_197,PAGE_VIEW,phone,598.31,2026-08-03T17:59:09.517Z
user_27,PAGE_VIEW,headphones,1311.87,2026-08-03T17:59:09.517Z
user_114,PURCHASE,laptop,397.79,2026-08-03T17:59:09.516Z
user_127,PAGE_VIEW,keyboard,132.12,2026-08-03T17:59:09.516Z
user_198,ADD_TO_CART,laptop,416.25,2026-08-03T17:59:09.516Z
user_71,PURCHASE,phone,279.32,2026-08-03T17:59:09.516Z
user_108,PURCHASE,headphones,321.37,2026-08-03T17:59:09.516Z
user_31,ADD_TO_CART,monitor,924.54,2026-08-03T17:59:09.516Z
